# Shoreline Transects Notebook
This notebook: reads either a single shapefile (with a year attribute) or multiple shapefiles (one per year), computes a baseline approximately parallel to shoreline orientation, generates transects spaced by a user-defined distance, finds transect × shoreline intersections, writes intersection points to a shapefile and CSV, and does not use ArcGIS.

## Requirements
Install the required packages in your environment if needed:
```bash
pip install geopandas shapely pyproj rtree fiona pandas numpy
````
: 
,
: { 
: 
 },
: [
,
,
,
,
,
,
,
,
,

: 
,
: { 
: 
 },
: [
    Read shorelines either from a directory of shapefiles (one per year) or a single shapefile.
    If `path` is a directory: reads each .shp and attempts to infer year from filename using `year_regex`.
    If `path` is a file: reads it and expects `year_field` to indicate the attribute with year.
    Returns a GeoDataFrame with a `year` column.
    """
    if os.path.isdir(path):
        gdfs = []
        for fname in sorted(os.listdir(path)):
            if fname.lower().endswith('.shp'):
                full = os.path.join(path, fname)
                g = gpd.read_file(full)
                m = re.search(year_regex, fname)
                year = m.group(1) if m else None
                if year is None:
                    # fallback to an incremental index if year not found
                    year = os.path.splitext(fname)[0]
                g = g.copy()
                g['year'] = year
                gdfs.append(g)
        if not gdfs:
            raise FileNotFoundError('No shapefiles found in directory: %s' % path)
        allg = pd.concat(gdfs, ignore_index=True)
        allg = gpd.GeoDataFrame(allg, geometry='geometry', crs=gdfs[0].crs)
        return allg
    else:
        g = gpd.read_file(path)
        if year_field is None:
            # try common names
            for candidate in ['year','Year','YEAR','Year_','YEAR_']:
                if candidate in g.columns:
                    year_field = candidate
                    break
        if year_field is None:
            raise ValueError('Single shapefile provided but `year_field` not given and not auto-detected.')
        g = g.copy()
        g['year'] = g[year_field].astype(str)
        return g

In [ ]:
def ensure_projected_gdf(gdf):
    """Return a copy in a suitable projected CRS (UTM) for distance computations.
    If already projected, returns gdf.copy().
    """
    if gdf.crs is None:
        raise ValueError('Input GeoDataFrame has no CRS. Please set `gdf.crs` before calling.')
    crs = CRS(gdf.crs)
    if crs.is_geographic:
        # compute UTM zone from centroid lon/lat
        centroid = gdf.unary_union.centroid
        lon, lat = centroid.x, centroid.y
        zone = int((lon + 180) / 6) + 1
        is_south = lat < 0
        epsg = 32700 + zone if is_south else 32600 + zone
        return gdf.to_crs(epsg=epsg)
    else:
        return gdf.copy()

In [ ]:
def compute_baseline_direction(gdf):
    """Compute dominant shoreline orientation (unit vector) using PCA on all shoreline coordinates.
    Returns (centroid_x, centroid_y), unit_direction_vector (dx,dy).
    """
    # collect all coordinates from lines (2D)
    pts = []
    for geom in gdf.geometry:
        if geom is None:
            continue
        if geom.geom_type == 'MultiLineString' or geom.geom_type == 'GeometryCollection':
            for part in geom:
                pts.extend(list(part.coords))
        elif geom.geom_type == 'LineString':
            pts.extend(list(geom.coords))
        else:
            try:
                # attempt to extract line-like coordinate sequences (e.g., polygons' exteriors)
                if geom.exterior is not None:
                    pts.extend(list(geom.exterior.coords))
            except Exception:
                pass
    pts = np.array(pts)[:,:2]  # drop any Z
    if len(pts) < 2:
        raise ValueError('Not enough points to determine orientation')
    # center
    mean = pts.mean(axis=0)
    X = pts - mean
    # SVD for principal direction
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    # first row of Vt is principal axis (direction of maximum variance)
    dir_vec = Vt[0, :2]
    # ensure unit length
    dir_vec = dir_vec / np.linalg.norm(dir_vec)
    return (mean[0], mean[1]), (float(dir_vec[0]), float(dir_vec[1]))

In [ ]:
def build_baseline(center, direction, length=10000):
    "Create a LineString baseline centered at `center` along `direction`.
    `length` is total length in map units.
    """
    cx, cy = center
    dx, dy = direction
    half = length / 2.0
    p1 = (cx - dx * half, cy - dy * half)
    p2 = (cx + dx * half, cy + dy * half)
    return LineString([p1, p2])

def generate_transects(baseline, spacing=50, transect_length=500):
    "Generate transects perpendicular to `baseline`.
    Returns list of (transect_id, LineString, baseline_point)
    """
    # sample baseline as a line and compute its direction
    bx, by = np.array(baseline.coords[0]), np.array(baseline.coords[-1])
    bvec = bx - by
    # baseline unit direction (dx,dy)
    dx, dy = bvec[0], bvec[1]
    L = math.hypot(dx, dy)
    ux, uy = dx / L, dy / L
    # perpendicular unit vector
    px, py = -uy, ux
    # create baseline sampling points at spacing
    nsteps = int(max(1, math.floor(L / spacing)))
    transects = []
    for i in range(nsteps + 1):
        t = i / max(1, nsteps)
        bx_i = bx[0] + (by[0] - bx[0]) * t
        by_i = bx[1] + (by[1] - bx[1]) * t
        # create transect centered at (bx_i,by_i) along perpendicular
        hl = transect_length / 2.0
        pstart = (bx_i - px * hl, by_i - py * hl)
        pend = (bx_i + px * hl, by_i + py * hl)
        line = LineString([pstart, pend])
        transects.append((i, line, (bx_i, by_i)))
    return transects

In [ ]:
def intersect_transects_shorelines(transects, shore_gdf):
    "For each transect, find intersections with shorelines and return a GeoDataFrame of points with attributes.
    Attributes: transect_id, year, baseline_x, baseline_y, distance_along_transect (from start),
    original_shore_index (index of shore_gdf row)
    """
    rows = []
    for tid, tline, baseline_pt in transects:
        for idx, shore in shore_gdf.iterrows():
            geom = shore.geometry
            inter = tline.intersection(geom)
            if inter.is_empty:
                continue
            # intersection can be Point, MultiPoint, or LineString (overlap)
            if inter.geom_type == 'Point':
                ips = [inter]
            elif inter.geom_type == 'MultiPoint' or inter.geom_type == 'GeometryCollection':
                ips = [g for g in inter.geoms if g.geom_type == 'Point']
            elif inter.geom_type in ('LineString', 'MultiLineString'):
                # overlapping segment - take its mid-point(s)
                if inter.geom_type == 'LineString':
                    ips = [inter.interpolate(0.5, normalized=True)]
                else:
                    ips = [seg.interpolate(0.5, normalized=True) for seg in inter.geoms]
            else:
                continue
            for p in ips:
                # compute distance along transect from its start
                start = Point(tline.coords[0])
                d = start.distance(p)
                rows.append({
                    'transect_id': int(tid),
                    'year': shore.get('year', None),
                    'baseline_x': float(baseline_pt[0]),
                    'baseline_y': float(baseline_pt[1]),
                    'distance': float(d),
                    'shore_index': int(idx),
                    'geometry': p
                })
    if not rows:
        return gpd.GeoDataFrame(columns=['transect_id','year','baseline_x','baseline_y','distance','shore_index','geometry'], geometry='geometry')
    out = gpd.GeoDataFrame(rows, geometry='geometry', crs=shore_gdf.crs)
    return out

## Example usage - set parameters and run the pipeline

In [ ]:
# USER PARAMETERS - adjust these to your data and preferences
# Either point `input_path` to a directory of shapefiles (one per year)
# or to a single shapefile and set `year_field` to the attribute name holding the year.
input_path = r'C:/path/to/shorelines_or_dir'  # <-- change this
year_field = None  # if single shapefile, set the field name, e.g. 'YEAR'
spacing = 50  # spacing between transects in map units (m if projected)
transect_length = 1000  # total length of each transect (m)
baseline_length = 20000  # total length of baseline used to place transects
output_shp = r'C:/path/to/output/transect_intersections.shp'
output_csv = r'C:/path/to/output/transect_intersections.csv'

# Load shorelines
shore_gdf = read_shorelines(input_path, year_field)
print('Read', len(shore_gdf), 'shoreline features')

# Project to UTM / metric if needed
proj_shore = ensure_projected_gdf(shore_gdf)

# compute baseline orientation
center, direction = compute_baseline_direction(proj_shore)
baseline = build_baseline(center, direction, length=baseline_length)

# generate transects
transects = generate_transects(baseline, spacing=spacing, transect_length=transect_length)
print('Generated', len(transects), 'transects')

# intersect
pts = intersect_transects_shorelines(transects, proj_shore)
print('Found', len(pts), 'intersection points')

# Save outputs: reproject points back to source CRS (if different) for shapefile
if proj_shore.crs != shore_gdf.crs:
    pts_out = pts.to_crs(shore_gdf.crs)
else:
    pts_out = pts.copy()

if len(pts_out):
    os.makedirs(os.path.dirname(output_shp), exist_ok=True)
    pts_out.to_file(output_shp)
    # write CSV with X/Y in output CRS
    df = pts_out.copy()
    df['x'] = df.geometry.x
    df['y'] = df.geometry.y
    df.drop(columns='geometry').to_csv(output_csv, index=False)
    print('Wrote:', output_shp, output_csv)
else:
    print('No intersections to write.')

## Notes and tips
- If your input shapefiles are in geographic coordinates (lon/lat), the notebook automatically reprojects to UTM for metric spacing.
- For a single shapefile containing many shorelines across years, set `year_field` to the attribute name with the year.
- For directory mode, filenames are scanned for a 4-digit year; change `year_regex` in `read_shorelines` if your naming differs.
- You can tune `spacing`, `transect_length`, and `baseline_length`.